# 03 - Explore data quality and initial metrics

Use SQL to inspect accepted orders, investigate data coverage and calculate delivered
item value by purchase month. These read-only Silver diagnostics precede the formal Gold tables.

**Run:** import this standalone notebook, select **Serverless**, keep the project's
catalog and choose **Run all**. No tables are written.


In [ ]:
import re
from pyspark.sql import functions as F

def quote_identifier(name):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', name):
        raise ValueError('Use an identifier containing letters, numbers and underscores: ' + repr(name))
    return '`' + name + '`'


def table_name(catalog, layer, entity):
    return '.'.join(quote_identifier(x) for x in [catalog, 'olist_' + layer, entity])


def config(spark, dbutils):
    default_catalog = spark.sql('SELECT current_catalog()').first()[0]
    try:
        catalog = dbutils.widgets.get('catalog').strip()
    except Exception:
        dbutils.widgets.text('catalog', default_catalog, 'Existing catalog')
        catalog = dbutils.widgets.get('catalog').strip()
    quote_identifier(catalog)
    return {'catalog': catalog, 'raw_path': '/Volumes/' + catalog + '/olist_landing/raw'}

In [ ]:
cfg = config(spark, dbutils)
orders = table_name(cfg['catalog'], 'silver', 'orders')
items = table_name(cfg['catalog'], 'silver', 'items')
payments = table_name(cfg['catalog'], 'silver', 'payments')
summary_name = table_name(cfg['catalog'], 'quality', 'load_summary')
orders_rejected = table_name(cfg['catalog'], 'quality', 'orders_rejected')
print('Catalog:', cfg['catalog'])

## 1. Reconcile quality counts

For each entity, explain the Bronze count using accepted rows, rejected rows and collapsed duplicates.
The summary describes the latest Silver run. If Bronze has since been reprocessed,
compare the summary with the current tables before relying on it.


In [ ]:
summary = spark.table(summary_name)
unbalanced = summary.filter(
    F.col('bronze_rows') != F.col('silver_rows') + F.col('rejected_rows') + F.col('duplicate_rows_collapsed')
)
if unbalanced.count():
    raise AssertionError('Row counts do not reconcile. Check the Silver run.')
display(summary.orderBy('entity'))
print('Row counts reconciled.')

## 2. Investigate quarantined orders

Eight delivered orders without delivery dates are separated under this project's policy.
Their associated items and payments are also quarantined to preserve Silver referential integrity.
The original records remain in Bronze; rejection does not prove that the transaction never happened.

Inspect the status, missing date, rejection reason and source file in the output.


In [ ]:
display(spark.sql(f'''
SELECT order_id, order_status, order_purchase_timestamp,
       order_delivered_customer_date, _rejection_reasons, _source_file
FROM {orders_rejected}
ORDER BY order_id
LIMIT 20
'''))

## 3. Inspect order statuses

Count each order once. Created, canceled, unavailable and in-transit orders should not
automatically enter a metric defined for delivered orders.


In [ ]:
display(spark.sql(f'''
SELECT order_status, COUNT(*) AS orders
FROM {orders}
GROUP BY order_status
ORDER BY orders DESC
'''))

## 4. Check the metric population

The monthly query includes delivered Silver orders with at least one valid item.
Count that population first. `item_orders` contains one row per order, preventing
multi-item orders from multiplying the order count.

This is coverage within Silver; rejected source records remain documented separately.


In [ ]:
display(spark.sql(f'''
WITH item_orders AS (
  SELECT DISTINCT order_id FROM {items}
)
SELECT COUNT(*) AS delivered_orders_in_silver,
       COUNT(i.order_id) AS delivered_orders_with_valid_items,
       COUNT(*) - COUNT(i.order_id) AS delivered_orders_without_valid_items
FROM {orders} o
LEFT JOIN item_orders i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
'''))

## 5. Calculate monthly delivered item value

- **Population:** delivered Silver orders with valid items.
- **Month:** purchase month from `order_purchase_timestamp`.
- **Amount:** sum of item `price` in BRL, excluding freight.
- **Average:** item value divided by the number of eligible orders.

This metric does not establish accounting revenue or a policy for refunds and revenue recognition.
Boundary months can be partial; an apparent decline in those months requires coverage checks.

The `order_values` CTE aggregates items by order before joining orders and summarizing by month.


In [ ]:
# Aggregate items by order BEFORE joining to avoid multiplied amounts.
# KPI: delivered item value, excluding freight; not accounting revenue.
display(spark.sql(f'''
WITH order_values AS (
 SELECT order_id, SUM(price) AS items_value, SUM(freight_value) AS freight_value
 FROM {items}
 GROUP BY order_id
)
SELECT date_format(o.order_purchase_timestamp, 'yyyy-MM') AS purchase_month,
 COUNT(*) AS delivered_orders_with_valid_items,
 ROUND(SUM(v.items_value), 2) AS delivered_items_value,
 ROUND(AVG(v.items_value), 2) AS average_items_value_per_order
FROM {orders} o
JOIN order_values v ON o.order_id = v.order_id
WHERE o.order_status = 'delivered'
GROUP BY date_format(o.order_purchase_timestamp, 'yyyy-MM')
ORDER BY purchase_month
'''))

## 6. Understand join multiplication

An order can have several items and payment components. Joining both tables directly
on `order_id` repeats each item for every payment component and each payment for every item.

The diagnostic below first aggregates each entity by order, then shows examples with
multiple payments and the duplication a direct join would produce. It is a diagnostic,
not a business KPI.


In [ ]:
display(spark.sql(f'''
WITH item_totals AS (
  SELECT order_id, COUNT(*) AS item_rows, SUM(price) AS correct_items_value
  FROM {items}
  GROUP BY order_id
), payment_totals AS (
  SELECT order_id, COUNT(*) AS payment_rows
  FROM {payments}
  GROUP BY order_id
)
SELECT i.order_id, i.item_rows, p.payment_rows,
       i.item_rows * p.payment_rows AS rows_if_direct_join,
       i.correct_items_value,
       i.correct_items_value * p.payment_rows AS repeated_items_value_if_direct_join
FROM item_totals i
JOIN payment_totals p ON i.order_id = p.order_id
WHERE p.payment_rows > 1 AND i.correct_items_value > 0
ORDER BY rows_if_direct_join DESC, i.order_id
LIMIT 10
'''))

## Interpretation checklist

1. Explain why Silver has fewer orders than Bronze.
2. Define `delivered_items_value` and its population precisely.
3. Explain why average order item value uses orders rather than item rows.
4. Show how a direct item/payment join can inflate totals.
5. State the metric's coverage and observed period.

Continue with `04_gold` to materialize the order fact and two summaries.
Incremental MERGE and orchestration remain future extensions.

Source: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce),
CC BY-NC-SA 4.0.


In [ ]:
print('Exploration complete. Review the monthly table in section 5.')